In [1]:
import numpy as np
from collections import namedtuple
from itertools import product
from runWidthCalc import getWidth
import tqdm

## New scan grid

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

mSs = [300.,500.,700.,900.,1100.] #mass of scalar particle [GeV]
dmSs =[10.,50.,100.,200.,400.] # mS - (m1+m2)
dm12 = 10.0
dm10s = [15.0,50.,100.,200.,400.]
ctaus = [10,20,50] #proper lifetime of Chi1 [m]
lambdaUV0 = 1e8
nWorkers = os.cpu_count() # number of parallel MadEvent runs

# 1) Build the list of valid points (no width computation yet)
grid = []
for mS, dmS, dm10, ctau in product(mSs, dmSs, dm10s, ctaus):
    m2 = float(round(((mS - dmS) + dm12)/2.,2))
    m1 = float(round(m2 - dm12,2))
    m0 = float(round(m1-dm10,2))
    if any(m < 0 for m in [mS, m1, m2, m0]):
        continue
    if sorted([m0,m1,m2,mS]) != [m0,m1,m2,mS]:
        continue
    if m1-m0 >= 250.: # Exclude points which allow chi1 -> chi0 + h + h
        continue
    grid.append((mS, m2, m1, m0, ctau))

# 2) Compute the widths for the unique (m1,m0) pairs in parallel.
# Each getWidth call runs MadEvent in its own temporary copy of the process folder,
# so the calls are independent. Threads are enough, since the work is done in a subprocess.
cache_width = {}
m1m0_points = sorted({(m1,m0) for _,_,m1,m0,_ in grid})
if not os.path.isdir('lifetime_scan'):
    # Generate the MadGraph process folder once (serially), before the parallel runs
    pt = m1m0_points.pop(0)
    cache_width[pt] = getWidth(*pt, lambdaUV0)
if m1m0_points:
    with ThreadPoolExecutor(max_workers=nWorkers) as executor:
        futures = {executor.submit(getWidth, m1, m0, lambdaUV0): (m1,m0) for m1,m0 in m1m0_points}
        for fut in tqdm.tqdm(as_completed(futures), total=len(futures)):
            key = futures[fut]
            try:
                cache_width[key] = fut.result()
            except Exception as e:
                print(f"Width calculation failed for (m1,m0)={key}: {e}")

# 3) Build the points
pts = []
for mS, m2, m1, m0, ctau in grid:
    if (m1,m0) not in cache_width:
        continue
    ctau0 = 1.97e-16/cache_width[(m1,m0)]
    lambdaUV = lambdaUV0*np.sqrt(ctau/ctau0)
    pt = [f"{x:1.4g}" if x < 1e4 else f"{x:1.4e}" for x in (mS, m2, m1, m0, lambdaUV, ctau)]
    pts.append(tuple(pt))

print(len(pts))

100%|██████████| 77/77 [17:38<00:00, 13.75s/it] 

267


In [4]:
print(str(pts).replace("'",""))

[(300, 150, 140, 125, 1.0121e+05, 10), (300, 150, 140, 125, 1.4313e+05, 20), (300, 150, 140, 125, 2.2631e+05, 50), (300, 150, 140, 90, 3.5995e+06, 10), (300, 150, 140, 90, 5.0904e+06, 20), (300, 150, 140, 90, 8.0487e+06, 50), (300, 150, 140, 40, 2.0208e+07, 10), (300, 150, 140, 40, 2.8579e+07, 20), (300, 150, 140, 40, 4.5187e+07, 50), (300, 130, 120, 105, 9.9708e+04, 10), (300, 130, 120, 105, 1.4101e+05, 20), (300, 130, 120, 105, 2.2295e+05, 50), (300, 130, 120, 70, 3.3748e+06, 10), (300, 130, 120, 70, 4.7728e+06, 20), (300, 130, 120, 70, 7.5464e+06, 50), (300, 130, 120, 20, 1.5188e+07, 10), (300, 130, 120, 20, 2.1479e+07, 20), (300, 130, 120, 20, 3.3961e+07, 50), (300, 105, 95, 80, 9.6948e+04, 10), (300, 105, 95, 80, 1.3710e+05, 20), (300, 105, 95, 80, 2.1678e+05, 50), (300, 105, 95, 45, 2.9468e+06, 10), (300, 105, 95, 45, 4.1675e+06, 20), (300, 105, 95, 45, 6.5893e+06, 50), (300, 55, 45, 30, 8.2121e+04, 10), (300, 55, 45, 30, 1.1614e+05, 20), (300, 55, 45, 30, 1.8363e+05, 50), (500, 

## Old grid

In [ ]:
mSs = [400, 450, 600, 800, 1000] #mass of scalar particle [GeV]
# beta1s = [0.1, 0.2, 0.25, 0.3, 0.4] #boost of the chi1 in the labframe
beta1s = [0.5,0.6,0.8] #boost of the chi1 in the labframe
Dm21s = [5] #mass difference between chi2 and chi1 [GeV]
Dm10s = [60, 100] #mass difference between chi1 and chi0 [GeV] (come back and try 100)
ctau0s = [20] #proper lifetime of Chi1 [m]


mSs = [400, 450, 600, 800, 1000] #mass of scalar particle [GeV]
# beta1s = [0.1, 0.2, 0.25, 0.3, 0.4] #boost of the chi1 in the labframe
beta1s = [0.5,0.6,0.8] #boost of the chi1 in the labframe
Dm21s = [5] #mass difference between chi2 and chi1 [GeV]
Dm10s = [60, 100] #mass difference between chi1 and chi0 [GeV] (come back and try 100)
ctau0s = [20] #proper lifetime of Chi1 [m]

In [ ]:
pts = []
lambdaUV0 = 9.07e6
for mS in mSs:
    for beta1 in beta1s:
        for Dm21 in Dm21s:
            m1_solutions = [ (-beta1 ** 2 * Dm21 ** 3 + sgn * (-beta1 ** 2 * Dm21 ** 4 * mS ** 2 + 2 * beta1 ** 2 * Dm21 ** 2 * mS ** 4 - beta1 ** 2 * mS ** 6 + Dm21 ** 4 * mS ** 2 - 2 * Dm21 ** 2 * mS ** 4 + mS ** 6) ** .5 + beta1 ** 2 * Dm21 * mS ** 2 + Dm21 ** 3 - Dm21 * mS ** 2)/(2 * (beta1 ** 2 * Dm21 ** 2 - Dm21 ** 2 + mS ** 2)) for sgn in [+1 , -1]]
            M1 = max(m1_solutions)
            M2 = M1 + Dm21 #mass of chi2 [GeV]
            for Dm10 in Dm10s:
                M0 = M1 - Dm10 #mass of chi0 [GeV]
                if sorted([mS,M2,M1,M0],reverse=True) != [mS,M2,M1,M0]:
                    print(f"Skipping point: mS={mS}, M2={M2}, M1={M1}, M0={M0} because it does not satisfy mS > M2 > M1 > M0")
                    continue
                if M1-M0 < 15.0:
                    continue
                p = [par for par in parameters if np.isclose(par.M1, M1,rtol=1e-3) and np.isclose(par.M0, M0,rtol=1e-3)]
                if not p:
                    print(f"Skipping point: mS={mS}, M2={M2}, M1={M1}, M0={M0} because it does not have a corresponding parameter file")
                    continue
                p = p[0]
                w10 = p.w1
                lambdaUV0 = p.lambdaUV
                for ctau0_m in ctau0s:
                    w1 = 6.58*3.e-17/ctau0_m
                    lambdaUV = lambdaUV0/np.sqrt(w1/w10)
                    # lambdaUV = lambdaUV0
                    pt = [float(f"{x:1.4g}") if x < 1e4 else float(f"{x:1.4e}") for x in (mS, M2, M1, M0, lambdaUV, ctau0_m)]
                    pts.append(tuple(pt))
                    

In [ ]:
print(len(pts))
print(pts)

In [ ]:
pts_m1 = []
for pt in pts:
    p = (pt[2],pt[3], pt[4])
    if p not in pts_m1:
        pts_m1.append(p)

In [ ]:
print(len(pts_m1))
print(pts_m1)